# Titanic Dataset – Data Visualization Project

**Goal:** Create clear, honest visualizations using a real passenger dataset and explain the main finding from each chart.

## Dataset
This project uses the public Titanic passenger dataset (891 records) from the seaborn-data GitHub repository. The dataset includes passenger class, sex, age, fare, family members aboard, port of embarkation, and survival status.

**Important:** Charts are generated from the actual CSV when the notebook is run. No chart values or findings are hard-coded.

## Questions explored
1. What does the age distribution look like?
2. How does age vary between passengers who survived and did not survive?
3. How are passenger fares distributed, and are there extreme values?
4. Is there a visible relationship between age and fare?
5. How does survival rate differ by passenger class and sex?
6. Which numeric variables show the strongest correlation with survival?

**Tools:** Python, Pandas, Matplotlib, Seaborn.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# A consistent, readable custom palette and plot style
sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"Did not survive": "#4477AA", "Survived": "#EE6677"}
ACCENT = "#228833"

# Create a folder for exported chart images
CHART_DIR = Path("charts")
CHART_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "savefig.dpi": 160,
    "savefig.bbox": "tight"
})

In [ ]:
# 2. Load the real Titanic dataset
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

In [ ]:
# 3. Prepare data for plotting
# Keep the original data unchanged; create a working copy.
data = df.copy()

# Friendly labels for charts
data["Survival"] = data["survived"].map({0: "Did not survive", 1: "Survived"})
data["Passenger class"] = data["pclass"].map({1: "1st class", 2: "2nd class", 3: "3rd class"})

# Median imputation is used only for numeric plotting columns where needed.
# This avoids silently dropping many passengers, while retaining a clear note.
data["age_for_plot"] = data["age"].fillna(data["age"].median())
data["fare_for_plot"] = data["fare"].fillna(data["fare"].median())

print(f"Rows: {len(data)}")
print(f"Missing age values: {data['age'].isna().sum()}")
print(f"Missing fare values: {data['fare'].isna().sum()}")
print("Age and fare medians are used only for plots that need complete numeric values.")

## 4. Histogram – Passenger age distribution

In [ ]:
fig, ax = plt.subplots()
sns.histplot(data=data, x="age", bins=25, color="#4477AA", edgecolor="white", ax=ax)
ax.set_title("Most recorded passengers were young or middle-aged adults")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of passengers")
ax.text(0.98, 0.95, f"Missing ages: {df['age'].isna().sum()}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9)
plt.tight_layout()
fig.savefig(CHART_DIR / "01_age_histogram.png")
plt.show()

median_age = df["age"].median()
print(f"Takeaway: The median recorded passenger age was {median_age:.1f} years; the histogram shows how ages are distributed around it.")

## 5. Box plot – Age by survival outcome

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=data, x="Survival", y="age_for_plot",
            order=["Did not survive", "Survived"],
            palette=PALETTE, showfliers=True, ax=ax)
ax.set_title("Age distributions overlap for survivors and non-survivors")
ax.set_xlabel("Survival outcome")
ax.set_ylabel("Age (years; missing values filled with median)")
plt.tight_layout()
fig.savefig(CHART_DIR / "02_age_boxplot_by_survival.png")
plt.show()

age_medians = data.groupby("Survival", observed=False)["age_for_plot"].median()
print(f"Takeaway: Median age was {age_medians.get('Survived', np.nan):.1f} years for survivors and {age_medians.get('Did not survive', np.nan):.1f} years for non-survivors; the boxes show spread and overlap.")

## 6. Box plot – Fare by survival outcome

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=data, x="Survival", y="fare_for_plot",
            order=["Did not survive", "Survived"],
            palette=PALETTE, showfliers=True, ax=ax)
ax.set_title("Survivors generally paid higher fares, with substantial overlap")
ax.set_xlabel("Survival outcome")
ax.set_ylabel("Fare (dataset currency units; missing values median-filled)")
plt.tight_layout()
fig.savefig(CHART_DIR / "03_fare_boxplot_by_survival.png")
plt.show()

fare_medians = data.groupby("Survival", observed=False)["fare_for_plot"].median()
print(f"Takeaway: Median fare was {fare_medians.get('Survived', np.nan):.2f} for survivors versus {fare_medians.get('Did not survive', np.nan):.2f} for non-survivors. This is an association, not proof that fare caused survival.")

## 7. Scatter plot – Age versus fare

In [ ]:
# Use original non-missing age/fare values so imputation does not create artificial points.
scatter_data = data.dropna(subset=["age", "fare"])
fig, ax = plt.subplots()
sns.scatterplot(data=scatter_data, x="age", y="fare", hue="Survival",
                hue_order=["Did not survive", "Survived"],
                palette=PALETTE, alpha=0.65, s=42, ax=ax)
ax.set_title("Age and fare show no simple linear pattern")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Fare")
ax.legend(title="Outcome")
plt.tight_layout()
fig.savefig(CHART_DIR / "04_age_vs_fare_scatter.png")
plt.show()
print(f"Takeaway: The scatter plot shows wide fare variation across ages; it does not suggest a simple age-to-fare relationship. Points shown: {len(scatter_data)}.")

## 8. Bar chart – Survival rate by passenger class

In [ ]:
class_rate = data.groupby("Passenger class", observed=False)["survived"].mean().reindex(
    ["1st class", "2nd class", "3rd class"])
fig, ax = plt.subplots()
bars = ax.bar(class_rate.index, class_rate.values, color=["#228833", "#66CCEE", "#AA3377"])
ax.set_title("Survival rate was highest in 1st class and lowest in 3rd class")
ax.set_xlabel("Passenger class")
ax.set_ylabel("Survival rate")
ax.set_ylim(0, 1)  # Starts at zero so bar lengths are not misleading.
ax.bar_label(bars, labels=[f"{v:.1%}" for v in class_rate.values], padding=3)
plt.tight_layout()
fig.savefig(CHART_DIR / "05_survival_rate_by_class.png")
plt.show()
print("Takeaway: " + "; ".join([f"{k}: {v:.1%}" for k, v in class_rate.items()]) + ".")

## 9. Bar chart – Survival rate by sex

In [ ]:
sex_rate = data.groupby("sex")["survived"].mean().reindex(["female", "male"])
fig, ax = plt.subplots()
bars = ax.bar(["Female", "Male"], sex_rate.values, color=["#EE6677", "#4477AA"])
ax.set_title("Women in this dataset had a higher survival rate than men")
ax.set_xlabel("Sex recorded in dataset")
ax.set_ylabel("Survival rate")
ax.set_ylim(0, 1)
ax.bar_label(bars, labels=[f"{v:.1%}" for v in sex_rate.values], padding=3)
plt.tight_layout()
fig.savefig(CHART_DIR / "06_survival_rate_by_sex.png")
plt.show()
print(f"Takeaway: Survival rate was {sex_rate['female']:.1%} for female passengers and {sex_rate['male']:.1%} for male passengers in this dataset.")

## 10. Correlation heatmap – Numeric variables

In [ ]:
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = data[numeric_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, linewidths=0.5, square=True, ax=ax)
ax.set_title("Passenger class and fare have the clearest numeric correlations with survival")
plt.tight_layout()
fig.savefig(CHART_DIR / "07_correlation_heatmap.png")
plt.show()

survival_corr = corr["survived"].drop("survived").sort_values(key=abs, ascending=False)
strongest = survival_corr.index[0]
print(f"Takeaway: Among the selected numeric variables, {strongest} has the largest absolute correlation with survival (r = {survival_corr.iloc[0]:.2f}). Correlation does not establish causation.")

## 11. Final summary

- Histograms show the distribution of a numeric variable.
- Box plots compare medians, spread, and possible outliers between groups.
- Scatter plots show the relationship between two numeric variables.
- Bar charts compare group-level rates; the vertical axis starts at zero.
- A correlation heatmap summarizes linear relationships among numeric variables.

### Honest interpretation notes
- This is an observational passenger dataset, so the charts show associations, not causes.
- Age and fare have missing values. Median-filled values are used only for the age/fare box plots; the scatter plot uses rows with both original values present.
- Correlation values depend on numeric encoding. `pclass` is an ordered category (1st, 2nd, 3rd), so interpret its correlation carefully.
- The dataset is a historical sample and should not be generalized to all ship disasters or modern travel.

### Exported files
All charts are saved as PNG files in the `charts/` folder when the notebook is run.